<a href="https://colab.research.google.com/github/FrankCRoth/colab_notebooks/blob/main/exam_prep_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cyber-Security Exam Prep — Colab edition

<a target="_blank" href="https://colab.research.google.com/github/FrankCRoth/colab_notebooks/blob/main/exam_prep_colab.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

Asks random questions from `question-catalog.txt` and grades your answers via an LLM through **OpenRouter**.

**One-time setup in Colab:**

1. Click the 🔑 **Secrets** icon in the left sidebar.
2. Add a secret named `OPENROUTER_API_KEY` with your key (`sk-or-v1-...`).
3. Toggle **Notebook access** ON for that secret.
4. Upload `question-catalog.txt` via the 📁 Files panel (or let the cell below prompt you).
5. Runtime → Run all.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Install dependencies (quiet)
!pip install -q openai ipywidgets

In [ ]:
import os
from google.colab import userdata

try:
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
    print("✓ OPENROUTER_API_KEY loaded from Colab Secrets")
except Exception as e:
    raise RuntimeError(
        "OPENROUTER_API_KEY not found in Colab Secrets.\n"
        "Open the 🔑 panel on the left, add the secret, and enable notebook access."
    ) from e

✓ OPENROUTER_API_KEY loaded from Colab Secrets


In [ ]:
# Load the question catalog. If the file isn't uploaded, you'll be prompted.
from pathlib import Path

CATALOG_PATH = Path("question-catalog.txt")

if not CATALOG_PATH.exists():
    from google.colab import files
    print("Please upload question-catalog.txt")
    uploaded = files.upload()
    name = next(iter(uploaded))
    if name != "question-catalog.txt":
        Path(name).rename(CATALOG_PATH)

QUESTIONS = [
    ln.strip()
    for ln in CATALOG_PATH.read_text(encoding="utf-8").splitlines()
    if ln.strip()
]
print(f"✓ Loaded {len(QUESTIONS)} questions from {CATALOG_PATH}")

✓ Loaded 50 questions from question-catalog.txt


In [ ]:
# Configure the OpenRouter client (OpenAI-compatible API)
from openai import OpenAI

MODEL = "openai/gpt-4o-mini"   # see https://openrouter.ai/models
# Alternatives:
#   "anthropic/claude-3.5-sonnet"
#   "meta-llama/llama-3.3-70b-instruct"
#   "google/gemini-2.0-flash-exp:free"   (free tier)

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

GRADER_SYSTEM = (
    "You are an expert cyber-security examiner grading a student's exam answer. "
    "Be fair, concise, and pedagogical.\n\n"
    "For each submission you receive a QUESTION and the STUDENT_ANSWER.\n"
    "Respond in Markdown with exactly these sections:\n"
    "**Score:** X / 10\n"
    "**Verdict:** correct | partially correct | incorrect\n"
    "**What was good:** ...\n"
    "**What is missing or wrong:** ...\n"
    "**Model answer:** a compact, exam-ready reference answer (5-10 sentences)."
)

def grade(question: str, answer: str) -> str:
    resp = client.chat.completions.create(
        model=MODEL,
        temperature=0.2,
        messages=[
            {"role": "system", "content": GRADER_SYSTEM},
            {"role": "user",
             "content": f"QUESTION:\n{question}\n\nSTUDENT_ANSWER:\n{answer}"},
        ],
    )
    return resp.choices[0].message.content or ""

# Quick sanity check
print("Testing model connectivity…")
print(client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Reply with the single word: ready."}],
    max_tokens=10,
).choices[0].message.content)

Testing model connectivity…
ready


## Interactive exam UI

Click **New question** to draw a random one, type your answer, then **Grade**.
**Show model answer** skips to the reference solution.

In [ ]:
import random
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

state = {"question": ""}

q_index   = widgets.HTML(value="")
q_label   = widgets.HTML(value="<i>Click 'New question' to start.</i>")
answer    = widgets.Textarea(
    placeholder="Type your answer here…",
    layout=widgets.Layout(width="100%", height="180px"),
)
btn_new   = widgets.Button(description="New question", button_style="primary")
btn_grade = widgets.Button(description="Grade", button_style="success")
btn_model = widgets.Button(description="Show model answer", button_style="warning")
btn_clear = widgets.Button(description="Clear")
out       = widgets.Output()

def on_new(_=None):
    idx = random.randrange(len(QUESTIONS))
    state["question"] = QUESTIONS[idx]
    q_index.value = f"<small>Question #{idx + 1} of {len(QUESTIONS)}</small>"
    q_label.value = (
        f"<div style='font-size:16px;line-height:1.5'>"
        f"<b>Q:</b> {QUESTIONS[idx]}</div>"
    )
    answer.value = ""
    with out:
        clear_output()

def on_grade(_=None):
    if not state["question"]:
        return
    if not answer.value.strip():
        with out:
            clear_output()
            print("Please type an answer first (or click 'Show model answer').")
        return
    with out:
        clear_output()
        print("Grading…")
    try:
        fb = grade(state["question"], answer.value.strip())
        with out:
            clear_output()
            display(Markdown(fb))
    except Exception as e:
        with out:
            clear_output()
            print(f"Error: {e}")

def on_model(_=None):
    if not state["question"]:
        return
    answer.value = "(no answer given — please provide a model answer)"
    on_grade()

def on_clear(_=None):
    answer.value = ""
    with out:
        clear_output()

btn_new.on_click(on_new)
btn_grade.on_click(on_grade)
btn_model.on_click(on_model)
btn_clear.on_click(on_clear)

ui = widgets.VBox([
    q_index,
    q_label,
    widgets.HBox([btn_new, btn_model]),
    widgets.HTML("<b>Your answer:</b>"),
    answer,
    widgets.HBox([btn_grade, btn_clear]),
    widgets.HTML("<hr><b>Feedback:</b>"),
    out,
])
display(ui)
on_new()